# Corpus analysis

Position statistics for a Corpus of Annotations. Every number here comes from the `bjj_analysis` module, which is tested on synthetic Corpora; this notebook only calls it and plots.

**Selection bias.** The Corpus covers only filmed, published Matches. Whole Brackets are annotated to avoid hand-picking, but events that are not filmed or not published are missing entirely.

## Load the Corpus

Annotations are read from `annotations/` at the repo root. Each file is checked against the Annotation schema and the vocabulary; files that fail are listed below instead of being used. Re-labels (`relabel_of` set) are left out. Set the metadata filters to stratify, e.g. `FILTERS = {"ruleset": "adcc", "gi": False}`.

While `annotations/` is empty, the notebook falls back to the fictional sample Annotation in `vocab/examples/` so every cell still runs.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

import bjj_analysis as bjj

REPO = Path.cwd().resolve().parent if Path.cwd().name == "analysis" else Path.cwd().resolve()
CORPUS_DIR = REPO / "annotations"
FILTERS = {}  # e.g. {"ruleset": "adcc", "gi": False, "weight_class": "-88kg"}

corpus = bjj.load_corpus(CORPUS_DIR, **FILTERS)
if corpus.matches.empty and corpus.rejected.empty:
    print(f"No Annotations in {CORPUS_DIR}; using the fictional sample instead.")
    corpus = bjj.load_corpus(REPO / "vocab" / "examples", pattern="sample-annotation.json", **FILTERS)

print(f"{len(corpus.matches)} Matches, {len(corpus.segments)} Segments, {len(corpus.rejected)} rejected files")

In [ ]:
corpus.rejected

In [ ]:
corpus.matches[["annotation_id", "event", "bracket", "round", "ruleset", "gi", "weight_class", "athlete_a", "athlete_b", "winner", "method"]]

## Next-Position distributions

For each Position, where earned Transitions lead next (first-order). Resets are path breaks and never count. `count` / `position_total` are the raw counts; `probability` is smoothed with a Dirichlet prior pooled by Family, so a Position seen only a few times borrows strength from its Family (`family_count` / `family_total`). Rows with `count` 0 are next Positions seen only from the Family.

In [ ]:
PRIOR_STRENGTH = 2.0         # α: pseudo-counts borrowed from the Family
FAMILY_PRIOR_STRENGTH = 1.0  # β: pseudo-counts spread evenly over every Position

next_positions = bjj.next_position_distribution(
    corpus, prior_strength=PRIOR_STRENGTH, family_prior_strength=FAMILY_PRIOR_STRENGTH
)
next_positions.round(3)

## Transition types into each next Position

In [ ]:
next_transitions = bjj.next_position_transitions(corpus)
next_transitions.round(3)

## Network graph

Nodes are Positions (coloured by Family, sized by Segment count); edges are earned Transitions, wider for more frequent ones.

In [ ]:
graph = bjj.transition_network(corpus)
fig, ax = plt.subplots(figsize=(12, 9))
bjj.plot_transition_network(graph, ax=ax)
ax.set_title("Earned Transitions between Positions")
plt.show()